In [1]:
import wisco_slap as wis

In [2]:
import os

In [3]:
si = wis.meta.get.sync_info()

In [ ]:
for subject in si.keys():
    for exp in si[subject].keys():
        try:
            wis.pns.eyes._run_pupil_inference(subject, exp, redo=False)
        except Exception as e:
            print(f"Error occurred while processing {subject}_{exp}: {e}")

In [ ]:
subjects = ['alnair', 'alnair', 'sargas']
exps = ['exp_1', 'exp_2', 'exp_3']
for subject, exp in zip(subjects, exps):
    wis.pns.sync_block_dat._run_pupil_inference(subject, exp, redo=True)

In [6]:
for subject in si.keys():
    for exp in si[subject].keys():
        for sb in si[subject][exp]['sync_blocks']:
            pupil_dir = os.path.join(wis.defs.anmat_root, subject, exp, 'sync_block_data', f'sync_block-{sb}', 'pupil')
            if os.path.exists(pupil_dir):
                has_mp4 = any(f.endswith('.mp4') for f in os.listdir(pupil_dir))
                if not has_mp4:
                    print(f"Missing .mp4 file in {pupil_dir}")
            else:
                continue

In [5]:
import shutil

DLC_TAG = "DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60"

for subject in si.keys():
    for exp in si[subject].keys():
        old_pupil_inf_dir = os.path.join(wis.defs.anmat_root, subject, exp, 'pupil_inference')
        if not os.path.exists(old_pupil_inf_dir):
            continue

        # move per-sync-block inference files to new location
        for sb in si[subject][exp]['sync_blocks']:
            sb_tag = f"pupil-{sb}{DLC_TAG}"
            matching_files = [f for f in os.listdir(old_pupil_inf_dir) if f.startswith(sb_tag)]
            if not matching_files:
                continue

            new_pupil_dir = os.path.join(
                wis.defs.anmat_root, subject, exp,
                "sync_block_data", f"sync_block-{sb}", "pupil",
            )
            os.makedirs(new_pupil_dir, exist_ok=True)

            for fname in matching_files:
                src = os.path.join(old_pupil_inf_dir, fname)
                dst = os.path.join(new_pupil_dir, fname)
                shutil.move(src, dst)
                print(f"  moved: {fname}")

        # move eye_metrics per sync block
        old_eye_dir = os.path.join(old_pupil_inf_dir, "eye_metrics")
        if os.path.exists(old_eye_dir):
            for sb in si[subject][exp]['sync_blocks']:
                em_name = f"eye_metrics-{sb}.parquet"
                src = os.path.join(old_eye_dir, em_name)
                if not os.path.exists(src):
                    continue
                new_eye_dir = os.path.join(
                    wis.defs.anmat_root, subject, exp,
                    "sync_block_data", f"sync_block-{sb}", "pupil", "eye_metrics",
                )
                os.makedirs(new_eye_dir, exist_ok=True)
                shutil.move(src, os.path.join(new_eye_dir, em_name))
                print(f"  moved: eye_metrics/{em_name}")

            # remove old eye_metrics dir if empty
            if not os.listdir(old_eye_dir):
                os.rmdir(old_eye_dir)
                print(f"  removed empty: {old_eye_dir}")

        # remove old pupil_inference dir if empty
        if not os.listdir(old_pupil_inf_dir):
            os.rmdir(old_pupil_inf_dir)
            print(f"  removed empty: {old_pupil_inf_dir}")
        else:
            remaining = os.listdir(old_pupil_inf_dir)
            print(f"  WARNING: {old_pupil_inf_dir} still has files: {remaining}")

  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60.csv
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60.h5
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_filtered.csv
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_filtered.h5
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_full.pickle
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_meta.pickle
  moved: pupil-1DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_p60_labeled.mp4
  moved: pupil-2DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60.csv
  moved: pupil-2DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60.h5
  moved: pupil-2DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_filtered.csv
  moved: pupil-2DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_filtered.h5
  moved: pupil-2DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60_full.pic

## Whisking migration: `exp_#/whisking/` → `sync_block_data/sync_block-#/whisking/`
Move files and drop the `-{sb}` suffix from filenames (directory encodes identity).

In [6]:
# Whisking migration: move from exp_#/whisking/ to sync_block_data/sync_block-#/whisking/
# Rename files to drop the -{sb} suffix

WHISKING_FILES = {
    "mask-{sb}.tif": "mask.tif",
    "video_frame_stack-{sb}.tif": "video_frame_stack.tif",
    "whisk_df-{sb}.parquet": "whisk_df.parquet",
}

for subject in si.keys():
    for exp in si[subject].keys():
        old_whisk_dir = os.path.join(wis.defs.anmat_root, subject, exp, "whisking")
        if not os.path.exists(old_whisk_dir):
            continue

        print(f"\n{subject}/{exp}:")
        for sb in si[subject][exp]["sync_blocks"]:
            new_whisk_dir = os.path.join(
                wis.defs.anmat_root, subject, exp,
                "sync_block_data", f"sync_block-{sb}", "whisking",
            )

            moved_any = False
            for old_name_template, new_name in WHISKING_FILES.items():
                old_name = old_name_template.format(sb=sb)
                src = os.path.join(old_whisk_dir, old_name)
                if not os.path.exists(src):
                    continue
                os.makedirs(new_whisk_dir, exist_ok=True)
                dst = os.path.join(new_whisk_dir, new_name)
                shutil.move(src, dst)
                print(f"  moved: {old_name} -> sync_block-{sb}/whisking/{new_name}")
                moved_any = True

            if not moved_any:
                print(f"  sync_block-{sb}: no whisking files found")

        # remove Thumbs.db if present
        thumbs = os.path.join(old_whisk_dir, "Thumbs.db")
        if os.path.exists(thumbs):
            os.remove(thumbs)
            print(f"  removed: Thumbs.db")

        # remove old whisking dir if empty
        remaining = os.listdir(old_whisk_dir)
        if not remaining:
            os.rmdir(old_whisk_dir)
            print(f"  removed empty: {old_whisk_dir}")
        else:
            print(f"  WARNING: {old_whisk_dir} still has files: {remaining}")


alkaid/exp_2:
  moved: mask-1.tif -> sync_block-1/whisking/mask.tif
  moved: video_frame_stack-1.tif -> sync_block-1/whisking/video_frame_stack.tif
  moved: whisk_df-1.parquet -> sync_block-1/whisking/whisk_df.parquet
  moved: mask-2.tif -> sync_block-2/whisking/mask.tif
  moved: video_frame_stack-2.tif -> sync_block-2/whisking/video_frame_stack.tif
  moved: whisk_df-2.parquet -> sync_block-2/whisking/whisk_df.parquet
  moved: mask-3.tif -> sync_block-3/whisking/mask.tif
  moved: video_frame_stack-3.tif -> sync_block-3/whisking/video_frame_stack.tif
  moved: whisk_df-3.parquet -> sync_block-3/whisking/whisk_df.parquet
  removed: Thumbs.db
  removed empty: /data/slap_analysis/analysis_materials/alkaid/exp_2/whisking

alkaid/exp_3:
  moved: mask-1.tif -> sync_block-1/whisking/mask.tif
  moved: video_frame_stack-1.tif -> sync_block-1/whisking/video_frame_stack.tif
  moved: mask-2.tif -> sync_block-2/whisking/mask.tif
  moved: video_frame_stack-2.tif -> sync_block-2/whisking/video_frame_s

## Eye metrics rename: `eye_metrics-{sb}.parquet` → `eye_metrics.parquet`
Files are already in the correct directory — just drop the sync block suffix from the filename.

In [7]:
# Rename eye_metrics-{sb}.parquet -> eye_metrics.parquet in-place

for subject in si.keys():
    for exp in si[subject].keys():
        for sb in si[subject][exp]["sync_blocks"]:
            eye_dir = os.path.join(
                wis.defs.anmat_root, subject, exp,
                "sync_block_data", f"sync_block-{sb}", "pupil", "eye_metrics",
            )
            old_name = f"eye_metrics-{sb}.parquet"
            new_name = "eye_metrics.parquet"
            src = os.path.join(eye_dir, old_name)
            dst = os.path.join(eye_dir, new_name)

            if os.path.exists(dst):
                print(f"  {subject}/{exp}/sync_block-{sb}: eye_metrics.parquet already exists")
                continue
            if not os.path.exists(src):
                continue

            os.rename(src, dst)
            print(f"  {subject}/{exp}/sync_block-{sb}: {old_name} -> {new_name}")

  alkaid/exp_2/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  alkaid/exp_2/sync_block-2: eye_metrics-2.parquet -> eye_metrics.parquet
  alkaid/exp_2/sync_block-3: eye_metrics-3.parquet -> eye_metrics.parquet
  alkaid/exp_3/sync_block-2: eye_metrics-2.parquet -> eye_metrics.parquet
  alkaid/exp_3/sync_block-3: eye_metrics-3.parquet -> eye_metrics.parquet
  avior/exp_1/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  avior/exp_2/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  kaus/exp_1/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  kaus/exp_2/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  kaus/exp_2/sync_block-2: eye_metrics-2.parquet -> eye_metrics.parquet
  menka/exp_1/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet
  sargas/exp_3/sync_block-1: eye_metrics-1.parquet -> eye_metrics.parquet


## Finish alkaid/exp_1 pupil_inference migration
This directory was not processed by the first cell (still has old `pupil_inference/` dir).

In [8]:
# Finish any remaining pupil_inference/ directories
# (alkaid/exp_1 was missed by the first cell)

DLC_TAG = "DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60"

for subject in si.keys():
    for exp in si[subject].keys():
        old_pupil_inf_dir = os.path.join(wis.defs.anmat_root, subject, exp, "pupil_inference")
        if not os.path.exists(old_pupil_inf_dir):
            continue

        print(f"\n{subject}/{exp}: found remaining pupil_inference/")

        # move DLC inference files per sync block
        for sb in si[subject][exp]["sync_blocks"]:
            sb_tag = f"pupil-{sb}{DLC_TAG}"
            matching_files = [
                f for f in os.listdir(old_pupil_inf_dir)
                if os.path.isfile(os.path.join(old_pupil_inf_dir, f)) and f.startswith(sb_tag)
            ]
            if not matching_files:
                continue

            new_pupil_dir = os.path.join(
                wis.defs.anmat_root, subject, exp,
                "sync_block_data", f"sync_block-{sb}", "pupil",
            )
            os.makedirs(new_pupil_dir, exist_ok=True)

            for fname in matching_files:
                src = os.path.join(old_pupil_inf_dir, fname)
                dst = os.path.join(new_pupil_dir, fname)
                shutil.move(src, dst)
                print(f"  moved DLC: {fname}")

        # move + rename eye_metrics
        old_eye_dir = os.path.join(old_pupil_inf_dir, "eye_metrics")
        if os.path.exists(old_eye_dir):
            for sb in si[subject][exp]["sync_blocks"]:
                old_em = os.path.join(old_eye_dir, f"eye_metrics-{sb}.parquet")
                if not os.path.exists(old_em):
                    continue
                new_eye_dir = os.path.join(
                    wis.defs.anmat_root, subject, exp,
                    "sync_block_data", f"sync_block-{sb}", "pupil", "eye_metrics",
                )
                os.makedirs(new_eye_dir, exist_ok=True)
                dst = os.path.join(new_eye_dir, "eye_metrics.parquet")
                shutil.move(old_em, dst)
                print(f"  moved + renamed: eye_metrics-{sb}.parquet -> eye_metrics.parquet")

            if not os.listdir(old_eye_dir):
                os.rmdir(old_eye_dir)

        # clean up old pupil_inference dir
        if not os.listdir(old_pupil_inf_dir):
            os.rmdir(old_pupil_inf_dir)
            print(f"  removed empty: {old_pupil_inf_dir}")
        else:
            remaining = os.listdir(old_pupil_inf_dir)
            print(f"  WARNING: {old_pupil_inf_dir} still has files: {remaining}")

## Verification scan
Check that everything conforms to `directory_structure.md`. Report any issues.

In [9]:
# Verification: check that the directory structure is correct
import re

DLC_TAG = "DLC_Resnet50_dlc_slap_pupilSep23shuffle0_snapshot_best-60"
issues = []
ok_count = 0

for subject in si.keys():
    for exp in si[subject].keys():
        exp_root = os.path.join(wis.defs.anmat_root, subject, exp)

        # --- check old directories are gone ---
        old_pupil_inf = os.path.join(exp_root, "pupil_inference")
        if os.path.exists(old_pupil_inf):
            issues.append(f"OLD DIR EXISTS: {old_pupil_inf}")

        old_whisking = os.path.join(exp_root, "whisking")
        if os.path.exists(old_whisking):
            issues.append(f"OLD DIR EXISTS: {old_whisking}")

        # --- check each sync block ---
        for sb in si[subject][exp]["sync_blocks"]:
            sb_root = os.path.join(exp_root, "sync_block_data", f"sync_block-{sb}")

            # pupil directory
            pupil_dir = os.path.join(sb_root, "pupil")
            if os.path.exists(pupil_dir):
                for f in os.listdir(pupil_dir):
                    fpath = os.path.join(pupil_dir, f)
                    if os.path.isdir(fpath):
                        continue  # skip subdirs (eye_metrics)
                    # DLC files: should start with "pupil-" and contain DLC_TAG
                    if DLC_TAG in f:
                        ok_count += 1
                    else:
                        # non-DLC file in pupil dir — check it's expected
                        if f == "frame_times.npy":
                            ok_count += 1
                        else:
                            issues.append(f"UNEXPECTED FILE: {os.path.join('pupil', f)} in {subject}/{exp}/sync_block-{sb}")

                # eye_metrics subdirectory
                eye_dir = os.path.join(pupil_dir, "eye_metrics")
                if os.path.exists(eye_dir):
                    for f in os.listdir(eye_dir):
                        if f == "eye_metrics.parquet":
                            ok_count += 1
                        elif re.match(r"eye_metrics-\d+\.parquet", f):
                            issues.append(f"OLD NAMING: {f} in {subject}/{exp}/sync_block-{sb}/pupil/eye_metrics/")
                        else:
                            issues.append(f"UNEXPECTED: {f} in {subject}/{exp}/sync_block-{sb}/pupil/eye_metrics/")

            # whisking directory
            whisk_dir = os.path.join(sb_root, "whisking")
            if os.path.exists(whisk_dir):
                expected_whisking = {"mask.tif", "video_frame_stack.tif", "whisk_df.parquet"}
                for f in os.listdir(whisk_dir):
                    if f in expected_whisking:
                        ok_count += 1
                    elif re.match(r"(mask|video_frame_stack|whisk_df)-\d+\.", f):
                        issues.append(f"OLD NAMING: {f} in {subject}/{exp}/sync_block-{sb}/whisking/")
                    else:
                        issues.append(f"UNEXPECTED: {f} in {subject}/{exp}/sync_block-{sb}/whisking/")

print("=" * 60)
print(f"VERIFICATION COMPLETE: {ok_count} files OK, {len(issues)} issues")
print("=" * 60)

if issues:
    print("\nISSUES FOUND:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\nAll files are in the correct location with correct naming.")

VERIFICATION COMPLETE: 150 files OK, 0 issues

All files are in the correct location with correct naming.
